In [31]:
import mappy as mp
import sys
import statistics
from dataclasses import dataclass, asdict
import random
import numpy as np
from rapidfuzz import process, fuzz

# --- CONFIGURATION ---
# Define your backbone section here (or load from file)
BACKBONE_SECTION = "AAGCAAGTAAAACCTCTACAAATGTGGTATTGGCCCATCTCTATCGGTATCGTAGCATAACCCCTTGGGGCCTCTAAACGGGTCTTGAGGGGTTTTTTGTGCCCCTCGGGCCGGATTGCTATCTACCGGCATTGGCGCAGAAAAAAATGCCTGATGCGACGCTGCGCGTCTTATACTCCCACATATGCCAGATTCAGCAACGGATACGGCTTCCCCAACTTGCCCACTTCCATACGTGTCCTCCTTACCAGAAATTTATCCTTAAGGTCGTCAGCTATCCTGCAGGCGATCTCTCGATTTCGATCAAGACATTCCTTTAATGGTCTTTTCTGGACACCACTAGGGGTCAGAAGTAGTTCATCAAACTTTCTTCCCTCCCTAATCTCATTGGTTACCTTGGGCTATCGAAACTTAATTAACCAGTCAAGTCAGCTACTTGGCGAGATCGACTTGTCTGGGTTTCGACTACGCTCAGAATTGCGTCAGTCAAGTTCGATCTGGTCCTTGCTATTGCACCCGTTCTCCGATTACGAGTTTCATTTAAATCATGTGAGCAAAAGGCCAGCAAAAGGCCAGGAACCGTAAAAAGGCCGCGTTGCTGGCGTTTTTCCATAGGCTCCGCCCCCCTGACGAGCATCACAAAAATCGACGCTCAAGTCAGAGGTGGCGAAACCCGACAGGACTATAAAGATACCAGGCGTTTCCCCCTGGAAGCTCCCTCGTGCGCTCTCCTGTTCCGACCCTGCCGCTTACCGGATACCTGTCCGCCTTTCTCCCTTCGGGAAGCGTGGCGCTTTCTCATAGCTCACGCTGTAGGTATCTCAGTTCGGTGTAGGTCGTTCGCTCCAAGCTGGGCTGTGTGCACGAACCCCCCGTTCAGCCCGACCGCTGCGCCTTATCCGGTAACTATCGTCTTGAGTCCAACCCGGTAAGACACGACTTATCGCCACTGGCAGCAGCCACTGGTAACAGGATTAGCAGAGCGAGGTATGTAGGCGGTGCTACAGAGTTCTTGAAGTGGTGGCCTAACTACGGCTACACTAGAAGAACAGTATTTGGTATCTGCGCTCTGCTGAAGCCAGTTACCTTCGGAAAAAGAGTTGGTAGCTCTTGATCCGGCAAACAAACCACCGCTGGTAGCGGTGGTTTTTTTGTTTGCAAGCAGCAGATTACGCGCAGAAAAAAAGGATCTCAAGAAGATCCTTTGATCTTTTCTACGGGGTCTGACGCTCAGTGGAACGAAAACTCACGTTAAGGGATTTTGGTCATGAGATTATCAAAAAGGATCTTCACCTAGATCCTTTTAAATTAAAAATGAAGTTTTAAATCAATCTAAAGTATATATGAGTAAACTTGGTCTGACAGTTACCAATGCTTAATCAGTGAGGCACCTATCTCAGCGATCTGTCTATTTCGTTCATCCATAGTTGCATTTAAATTTCCGAACTCTCCAAGGCCCTCGTCGGAAAATCTTCAAACCTTTCGTCCGATCCATCTTGCAGGCTACCTCTCGAACGAACTATCGCAAGTCTCTTGGCCGGCCTTGCGCCTTGGCTATTGCTTGGCAGCGCCTATCGCCAGGTATTACTCCAATCCCGAATATCCGAGATCGGGATCACCCGAGAGAAGTTCAACCTACATCCTCAATCCCGATCTATCCGAGATCCGAGGAATATCGAAATCGGGGCGCGCCTGGTGTACCGAGAACGATCCTCTCAGTGCGAGTCTCGACGATCCATATCGTTGCTTGGCAGTCAGCCAGTCGGAATCCAGCTTGGGACCCAGGAAGTCCAATCGTCAGATATTGTACTCAAGCCTGGTCACGGCAGCGTACCGATCTGTTTAAACCTAGATATTGATAGTCTGATCGGTCAACGTATAATCGAGTCCTAGCTTTTGCAAACATCTATCAAGAGACAGGATCAGCAGGAGGCTTTCGCATGAGTATTCAACATTTCCGTGTCGCCCTTATTCCCTTTTTTGCGGCATTTTGCCTTCCTGTTTTTGCTCACCCAGAAACGCTGGTGAAAGTAAAAGATGCTGAAGATCAGTTGGGTGCGCGAGTGGGTTACATCGAACTGGATCTCAACAGCGGTAAGATCCTTGAGAGTTTTCGCCCCGAAGAACGCTTTCCAATGATGAGCACTTTTAAAGTTCTGCTATGTGGCGCGGTATTATCCCGTATTGACGCCGGGCAAGAGCAACTCGGTCGCCGCATACACTATTCTCAGAATGACTTGGTTGAGTATTCACCAGTCACAGAAAAGCATCTTACGGATGGCATGACAGTAAGAGAATTATGCAGTGCTGCCATAACCATGAGTGATAACACTGCGGCCAACTTACTTCTGACAACGATTGGAGGACCGAAGGAGCTAACCGCTTTTTTGCACAACATGGGGGATCATGTAACTCGCCTTGATCGTTGGGAACCGGAGCTGAATGAAGCCATACCAAACGACGAGCGTGACACCACGATGCCTGTAGCAATGGCAACAACCTTGCGTAAACTATTAACTGGCGAACTACTTACTCTAGCTTCCCGGCAACAGTTGATAGACTGGATGGAGGCGGATAAAGTTGCAGGACCACTTCTGCGCTCGGCCCTTCCGGCTGGCTGGTTTATTGCTGATAAATCTGGAGCCGGTGAGCGTGGGTCTCGCGGTATCATTGCAGCACTGGGGCCAGATGGTAAGCCCTCCCGTATCGTAGTTATCTACACGACGGGGAGTCAGGCAACTATGGATGAACGAAATAGACAGATCGCTGAGATAGGTGCCTCACTGATTAAGCATTGGTAACCGATTCTAGGTGCATTGGCGCAGAAAAAAATGCCTGATGCGACGCTGCGCGTCTTATACTCCCACATATGCCAGATTCAGCAACGGATACGGCTTCCCCAACTTGCCCACTTCCATACGTGTCCTCCTTACCAGAAATTTATCCTTAAGATCCCGAATCGTTTAAACTCGACTCTGGCTCTATCGAATCTCCGTCGTTTCGAGCTTACGCGAACAGCCGTGGCGCTCATTTGCTCGTCGGGCATCGAATCTCGTCAGCTATCGTCAGCTTACCTTTTTGGCAGCGATCGCGGCTCCCGACATCTTGGACCATTAGCTCCACAGGTATCTTCTTCCCTCTAGTGGTCATAACAGCAGCTTCAGCTACCTCTCAATTCAAAAAACCCCTCAAGACCCGTTTAGAGGCCCCAAGGGGTTATGCTATCAATCGTTGCGTTACACACACAAAAAACCAACACACATCCATCTTCGATGGATAGCGATTTTATTATCTAACTGCTGATCGAGTGTAGCCAGATCTAGTAATCAATTACGGGGTCATTAGTTCATAGC".upper()

@dataclass
class AlignmentStats:
    read_id: str
    read_len: int
    ref_start: int
    ref_end: int
    strand: int      # 1 for forward, -1 for reverse
    matches: int     # Number of matching bases
    block_len: int   # Length of the alignment block on the reference
    nm: int          # Edit distance (mismatches + gaps)
    error_rate: float
    is_valid: bool   # Flag if you want to filter later (e.g. alignment too short)

def estimate_error_rates(fastq_path, reference_seq, min_length=3000, min_coverage_fraction=0.95, max_indel=10):
    """
    Aligns reads to a specific backbone sequence and calculates error rates.
    """
    print(f"Building index for reference ({len(reference_seq)} bp)...", file=sys.stderr)
    
    # 1. Create Aligner from the string directly
    # 'preset="map-ont"' is optimized for Nanopore reads
    aligner = mp.Aligner(seq=reference_seq, preset="map-ont")
    
    if not aligner:
        raise ValueError("Failed to build index. Is the sequence empty?")

    stats_storage = []
    
    print(f"Processing {fastq_path}...", file=sys.stderr)

    extracted_backbones = []
    ref_len = len(reference_seq)
    
    for name, seq, qual in mp.fastx_read(fastq_path):
        if len(seq) < min_length:
            continue
        
        # Align read to the backbone section
        # mappy.map() returns a generator of alignments
        hits = list(aligner.map(seq))
        
        if not hits:
            continue

        # 2. Find Best Single Alignment
        # Strategy: Sort by 'mlen' (matched length) descending. 
        # The hit with the most matching bases is almost always the "primary" alignment.
        best_hit = sorted(hits, key=lambda x: x.mlen, reverse=True)[0]

        # Check mapping criteria
        if best_hit.cigar:
            # 1. Start/End Check (Tolerance 2 bp)
            starts_correctly = best_hit.r_st <= 2
            ends_correctly   = best_hit.r_en >= (ref_len - 2)

            # 2. Internal Coverage Check (>= 95%)
            # Sum Ops: 0 (Match/Mismatch), 7 (Equal), 8 (Diff)
            aligned_bases = sum(length for length, op in best_hit.cigar if op in (0, 7, 8))
            has_coverage  = (aligned_bases / ref_len) >= 0.95

            # 3. Max Indel Check (No indel > X bases)
            # Check Ops: 1 (Insertion) and 2 (Deletion)
            # We default to 0 in case there are no indels at all
            largest_indel = max([length for length, op in best_hit.cigar if op in (1, 2)], default=0)
            
            # COMBINE ALL CHECKS
            if starts_correctly and ends_correctly and has_coverage and (largest_indel <= max_indel):
                
                read_segment = seq[best_hit.q_st : best_hit.q_en]
                
                if best_hit.strand == -1:
                    read_segment = mp.revcomp(read_segment)
                    
                extracted_backbones.append(read_segment)
        
        # 3. Calculate Stats
        # nm = Edit Distance (mismatches + insertions + deletions)
        # blen = Alignment block length on the reference
        # Error Rate = Edit Distance / (Reference Span)
        # (You could also use seq length, but ref span is standard for 'divergence')
        
        if best_hit.blen == 0: continue

        err_rate = best_hit.NM / best_hit.blen
        
        # Store in RAM
        stat = AlignmentStats(
            read_id=name,
            read_len=len(seq),
            ref_start=best_hit.r_st,
            ref_end=best_hit.r_en,
            strand=best_hit.strand,
            matches=best_hit.mlen,
            block_len=best_hit.blen,
            nm=best_hit.NM,
            error_rate=err_rate,
            is_valid=True
        )
        
        stats_storage.append(stat)

    return stats_storage, extracted_backbones

# --- ANALYSIS HELPER ---
def analyze_results(stats_list):
    if not stats_list:
        print("No alignments found.")
        return

    # Filter for valid alignments only
    valid_stats = [s for s in stats_list if s.is_valid]
    
    if not valid_stats:
        print("No valid alignments passed the length filter.")
        return

    # Extract error rates
    errors = [s.error_rate for s in valid_stats]
    
    print("\n--- ERROR RATE ANALYSIS ---")
    print(f"Total Reads Aligned: {len(stats_list)}")
    print(f"Valid Alignments (>200bp): {len(valid_stats)}")
    print(f"Mean Error Rate:   {statistics.mean(errors):.2%}")
    print(f"Median Error Rate: {statistics.median(errors):.2%}")
    print(f"Min Error Rate:    {min(errors):.2%}")
    print(f"Max Error Rate:    {max(errors):.2%}")



# --- EXAMPLE USAGE ---
if __name__ == "__main__":
    # Example Inputs
    fastq_file = "/Users/ogw/Downloads/ris_plasmids/no_sample_id/20251208_1553_MN41644_AYO707_6ec906fc/fastq_pass/combined.fastq.gz"
    fastq_file = '/Users/ogw/Library/CloudStorage/GoogleDrive-oscargwilkins@gmail.com/My Drive/UCL PhD/2025/plasmid_sequencing_results/21641/2025-10-30_01-47-43/downstream_risdiplam_array_pool/downstream_risdiplam_array_pool_raw.fastq.gz'
    # Run
    stats, backbones = estimate_error_rates(fastq_file, BACKBONE_SECTION)
    analyze_results(stats)

    if backbones:
        # 1. Randomly sample 1000 if necessary
        sample_size = min(len(backbones), 1000)
        sampled_backbones = random.sample(backbones, sample_size)
        
        print(f"Calculating similarity matrix for {sample_size} sequences...")

        # Prepare data: cdist expects 2D arrays (list of lists)
        backbone_ratios = process.cdist(sampled_backbones, sampled_backbones, scorer=fuzz.ratio, dtype=np.float32, workers=-1)
        # Extract upper triangle (k=1 excludes diagonal) to get unique pairwise scores
        flat_ratios = backbone_ratios[np.triu_indices_from(backbone_ratios, k=1)]
        
        # Now you can visualize it (e.g., plt.hist(flat_ratios))


        

    

Building index for reference (3353 bp)...
Processing /Users/ogw/Library/CloudStorage/GoogleDrive-oscargwilkins@gmail.com/My Drive/UCL PhD/2025/plasmid_sequencing_results/21641/2025-10-30_01-47-43/downstream_risdiplam_array_pool/downstream_risdiplam_array_pool_raw.fastq.gz...



--- ERROR RATE ANALYSIS ---
Total Reads Aligned: 1456
Valid Alignments (>200bp): 1456
Mean Error Rate:   1.47%
Median Error Rate: 0.95%
Min Error Rate:    0.00%
Max Error Rate:    15.03%
Calculating similarity matrix for 342 sequences...


In [32]:
flat_ratios

array([98.30661, 98.39556, 98.45693, ..., 98.99866, 97.17108, 97.2139 ],
      shape=(58311,), dtype=float32)

In [39]:
np.percentile(flat_ratios, 1)

np.float32(93.60345)